In [ ]:
import win32com.client # for outlook
import re # for regex
import base64
import time
import requests #for sending the data to the virustotal API

API_KEY = "Removing API key for security reasons" #fetch API from virustotal profile.

VirusTotal = {
    "accept": "application/json",
    "x-apikey": API_KEY
}

# Property tag for raw Internet Headers
HEADER_PROP = "http://schemas.microsoft.com/mapi/proptag/0x007D001E"

# Connect to the local Outlook desktop application
outlook = win32com.client.Dispatch("Outlook.Application").GetNamespace("MAPI")

# Folder number 6 is the default Inbox
inbox = outlook.GetDefaultFolder(6)

# Grab all items in the Inbox
messages = inbox.Items

# Sort the items by the received date in descending order (newest first)
messages.Sort("[ReceivedTime]", True)

In [ ]:
def reputation_check(indicator):
    
    for type in indicator.keys():
        
        if type == "urls":
            for url in indicator[type]:
                print("*"*100)
                
                #create the base url for the virustotal API
                virus_total_url = "https://www.virustotal.com/api/v3/"

                #append the type of indicator to the url
                virus_total_url += type.lower() + "/"                   
                
                #encoding url to base 64
                url_id = base64.urlsafe_b64encode(url.strip().encode()).decode().strip("=") 
                virus_total_url += url_id

                #send the request to the virustotal API with the url and the headers containing the API key
                response = requests.get(virus_total_url, headers=VirusTotal) 

                if response.status_code == 200:
                    stats = response.json()['data']['attributes']['last_analysis_stats']
                    print(f"[{type}] {url} \n-> Malicious: {stats['malicious']} | Suspicious: {stats['suspicious']} | Harmless: {stats['harmless']}")
                elif response.status_code == 404:
                    print(f"[{type}] {url} \n-> Not found in VirusTotal database")
                else:
                    print(f"[{type}] {url} \n-> Error {response.status_code}: {response.json().get('error', {}).get('message')}")
                
                time.sleep(15) #Request rate: 4 lookups / min
        
        elif type == "ip_addresses":
            for ip in indicator[type]:
                print("*"*100)
                virus_total_url = "https://www.virustotal.com/api/v3/"
                virus_total_url += type.lower() + "/"
                virus_total_url += ip
                response = requests.get(virus_total_url, headers=VirusTotal)

                if response.status_code == 200:
                    stats = response.json()['data']['attributes']['last_analysis_stats']
                    print(f"[{type}] {ip} \n-> Malicious: {stats['malicious']} | Suspicious: {stats['suspicious']} | Harmless: {stats['harmless']}")
                elif response.status_code == 404:
                    print(f"[{type}] {ip} \n-> Not found in VirusTotal database")
                else:
                    print(f"[{type}] {ip} \n-> Error {response.status_code}: {response.json().get('error', {}).get('message')}")
                
                time.sleep(15)

        elif type == "domains":
            for domain in indicator[type]:
                print("*"*100)
                virus_total_url = "https://www.virustotal.com/api/v3/"
                virus_total_url += type.lower() + "/"
                virus_total_url += domain
                response = requests.get(virus_total_url, headers=VirusTotal)

                if response.status_code == 200:
                    stats = response.json()['data']['attributes']['last_analysis_stats']
                    print(f"[{type}] {domain} \n-> Malicious: {stats['malicious']} | Suspicious: {stats['suspicious']} | Harmless: {stats['harmless']}")
                elif response.status_code == 404:
                    print(f"[{type}] {domain} \n-> Not found in VirusTotal database")
                else:
                    print(f"[{type}] {domain} \n-> Error {response.status_code}: {response.json().get('error', {}).get('message')}")
                
                time.sleep(15)

       # response = requests.get(virus_total_url, headers=VirusTotal)
    return None


def main():
    counter = 0
    for message in messages:
        # Print the subject and received time of each email
        if counter == 1:
            break
        counter += 1
        #print(f"Subject: {message.Subject} \nReceived: {message.ReceivedTime}\nFrom: {message.SenderEmailAddress}\nBody: {message.Body} \nAttachments: {message.Attachments} ")
        headers = message.PropertyAccessor.GetProperty(HEADER_PROP)
        #print(headers)

        # Extracting sender email address, domain, sender IP, return path, subject, authentication protocols, and action from the email headers

        sender_address = message.senderEmailAddress
        domain_address = sender_address.split("@")[-1]
        sender_ip = (re.search(r'X-Sender-IP: [\d\.]+', headers).group(0)).split(": ")[1]
        return_path = (re.search(r'Reply-To:\s*(.*?)(?=[A-Z][a-zA-Z0-9-]*:|\r|\n|$)', headers).group(0)).split(": ")[1]
        subject = message.Subject
        
        Authentication = {}
        Authentication ["spf"] = (re.search(r'spf=[A-Za-z]+', headers).group(0)).split("=")[1]
        Authentication ["dkim"] = (re.search(r'dkim=[A-Za-z]+', headers).group(0)).split("=")[1]
        Authentication ["dmarc"] = (re.search(r'dmarc=[A-Za-z]+', headers).group(0)).split("=")[1]
        action = (re.search(r'action=[A-Za-z]+', headers).group(0)).split("=")[1]
        
        body = message.Body
        URLs = re.findall(r'(http[s]?://[^\s]+)', body)

        Indicators = {}
        Indicators["urls"] = URLs
        Indicators["ip_addresses"] = [sender_ip]
        Indicators["domains"] = [domain_address]
        
        print("sender_address = ", sender_address)
        print("domain_address = ", domain_address)
        print("sender_ip = ", sender_ip)
        print("return_path = ", return_path)
        print("subject = ", subject)
        print("Authentication Protocols = ", Authentication)
        print("action = ", action)

        #print(Indicators)
        print("=" * 100)  # Print a separator line between emails
        reputation_check(Indicators)
    
    

if __name__ == "__main__":
    main()

sender_address =  linkedin@em.linkedin.com
domain_address =  em.linkedin.com
sender_ip =  130.248.165.121
return_path =  "LinkedIn" <donotreply@em.linkedin.com>
subject =  The AI Opportunity Gap
Authentication Protocols =  {'spf': 'pass', 'dkim': 'pass', 'dmarc': 'pass'}
action =  none
****************************************************************************************************
[urls] https://t1.em.linkedin.com/r/?id=h4247b523-1e16-4870-bdf7-b6edd3cf32cd,af6434,d26694&e=bWNpZD03NTAzMTI3NjU3NTAxNDg3MTA0JnRyaz1lbWwtbWt0Zy1jZ2ktYnJhbmQtYXdyLWVnLWFpLW9wcG9ydHVuaXR5LWdhcC1yZXNlYXJjaCZzcmM9ZS1lbWw&s=5oqV27Hunyg3685IEc7VJ_QkxEkzdUloGwYW4iYf91g> 
-> Not found in VirusTotal database
****************************************************************************************************
[urls] https://t1.em.linkedin.com/r/?id=h4247b523-1e16-4870-bdf7-b6edd3cf32cd,af6434,d26696&e=bWNpZD03NTAzMTI3NjU3NTAxNDg3MTA0JnRyaz1lbWwtbWt0Zy1jZ2ktYnJhbmQtYXdyLWVnLWFpLW9wcG9ydHVuaXR5LWdhcC1yZXNlYXJjaCZzcmM